[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powerzoojax/PowerZooPy/blob/main/docs/en/examples/notebooks/nb06_task_ev.ipynb)

# NB06 — Task 3: EV V2G

> **Prerequisites**: [NB05 — Task 2: DER Arbitrage](./nb05_task_der.ipynb)  
> **Time**: ~10 minutes

## What You'll Learn

1. What the **availability mask** is and why it makes Task 3 unique
2. How commute schedules create intermittent agent availability
3. Gantt-chart visualization of EV presence windows
4. The **departure SOC constraint** as a hard safety requirement
5. `departure_success_rate` as an additional benchmark metric

## Background: What Makes Task 3 Distinctive

Task 3 inherits Task 2's temporal planning challenge (persistent SOC) and adds:

**1. Intermittent availability**: EVs are away on commute trips. When away, the agent
cannot charge/discharge — its action is **masked to zero** regardless of what the
policy outputs.

**2. Hard departure constraint**: at the moment of departure, the EV must have
SOC ≥ 80%. Failing this is penalized heavily (`departure_penalty_weight = 5.0`).

**ML analogy**: this is like a robotics task where the robot is periodically
"unavailable" (end-effector in contact with object), and has a **deadline constraint**
that must be satisfied before each "unavailable" period.

| Feature | Task 2 (Battery) | Task 3 (EV) |
|---|---|---|
| Agent availability | Always | **Intermittent (commute trips)** |
| Action masking | None | **Masked when away** |
| Departure constraint | None | **SOC ≥ 80% required at each departure** |
| Episode length | 48 steps (1 day) | **168 steps (1 week, hourly)** |
| Extra metric | — | **departure_success_rate** |

## Setup

In [ ]:
import powerzoo
print(f"PowerZoo version: {powerzoo.__version__}")

In [ ]:
from powerzoo.tasks import make_task_env
from powerzoo.wrappers import GymnasiumWrapper
from powerzoo.benchmarks.policies import RandomPolicy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Task Configuration

In [ ]:
from powerzoo.tasks.simple.task3_marl_ev import MARLEVTask

task = MARLEVTask(split='train')
scenario = task.get_scenario_config()

print("=== Task 3: marl_ev_v2g ===")
print(f"Grid:          Case33 — IEEE 33-bus distribution")
print(f"Num EVs:       {task._num_evs}")
print(f"EV capacity:   {task._ev_capacity_kwh} kWh each")
print(f"Charge power:  {task._ev_charge_power_kw} kW")
print(f"Discharge (V2G): {task._ev_discharge_power_kw} kW")
print(f"Episode length: {task._max_steps} steps = 1 week @ 1h intervals")
print(f"SOC departure min: {task._soc_departure_min} (must reach before leaving)")
print(f"\nReward weights:")
r = scenario['reward']
print(f"  Arbitrage:        {r['arbitrage_weight']}")
print(f"  Departure penalty: {r['departure_penalty_weight']}")
print(f"  Home violation:   {r['home_violation_penalty']}")

## 2. Commute Schedules

Each EV follows a daily commute pattern. The schedule defines when the EV departs,
when it returns, and how much energy was consumed during the trip.

In [ ]:
schedules = task._get_default_commute_schedules()

print("Default commute schedules (per day):")
for i, schedule in enumerate(schedules):
    print(f"\n  EV {i}:")
    for trip in schedule:
        print(f"    Depart {trip['departure']:.0f}:00 → Return {trip['arrival']:.0f}:00  "
              f"(consume {trip['energy_kWh']:.0f} kWh)")

## 3. Availability Window Visualization (Gantt Chart)

The following chart shows when each EV is **at home** (can charge/discharge)
vs **away** (action masked). This is the availability mask.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

colors = ['#1565C0', '#E65100', '#2E7D32', '#6A1B9A', '#00695C']
n_days = 7
n_evs = len(schedules)

for ev_idx, schedule in enumerate(schedules):
    y = ev_idx
    color = colors[ev_idx % len(colors)]

    for day in range(n_days):
        day_offset = day * 24
        # Mark home windows (complement of away windows)
        away_periods = [(t['departure'] + day_offset, t['arrival'] + day_offset)
                        for t in schedule]

        # Draw full day as home (light color)
        ax.barh(y, 24, left=day_offset, height=0.5, color=color, alpha=0.2,
                label=f'EV {ev_idx} home' if day == 0 else '')

        # Overlay away periods (dark)
        for depart, arrive in away_periods:
            duration = arrive - depart
            ax.barh(y, duration, left=depart, height=0.5, color='#777',
                    label='Away (action masked)' if day == 0 and ev_idx == 0 else '',
                    hatch='//', alpha=0.8)

        # Mark departure deadlines
        for t in schedule:
            ax.axvline(t['departure'] + day_offset, color=color, linewidth=0.8,
                       linestyle='--', alpha=0.5)

# Grid lines for days
for d in range(n_days + 1):
    ax.axvline(d * 24, color='gray', linewidth=0.5, alpha=0.4)

ax.set_yticks(range(n_evs))
ax.set_yticklabels([f'EV {i}' for i in range(n_evs)])
ax.set_xticks([d * 24 for d in range(n_days + 1)])
ax.set_xticklabels([f'Day {d+1}' for d in range(n_days)] + [''])
ax.set_xlabel('Time (hours into episode)')
ax.set_title('Task 3 — EV availability windows (light = home, hatched = away / action masked)')

# Legend
home_patch = mpatches.Patch(color='steelblue', alpha=0.4, label='At home (can charge/discharge)')
away_patch = mpatches.Patch(facecolor='#777', hatch='//', alpha=0.8, label='Away (action = 0)')
ax.legend(handles=[home_patch, away_patch], loc='upper right')

plt.tight_layout()
plt.show()

## 4. The Availability Mask in Practice

The `is_home` flag is included in each agent's local observation.
When `is_home = 0`, the environment ignores the agent's action (clips to 0).

In [ ]:
env = make_task_env("marl_ev_v2g", split="train")
obs_dict, info = env.reset(seed=42)

print("=== Observation structure for one EV ===")
agent = env.agents[0]
obs = obs_dict[agent]
print(f"Agent: {agent}")
print(f"Obs shape: {obs.shape}")
print(f"\nLocal obs fields (from task config):")
print("  - soc                (float)  current state of charge [0, 1]")
print("  - is_home            (float)  1.0 = at home, 0.0 = away (MASK)")
print("  - departure_ready    (float)  1.0 = SOC ≥ departure minimum")
print("  - power_limits       (float)  current max charge/discharge power")
print("  - time_to_departure  (float)  hours until next departure")
print("  - soc_departure_min  (float)  constant = 0.8")
print(f"\nActual obs values: {obs}")

In [ ]:
# Simulate is_home evolution over one week
env2 = make_task_env("marl_ev_v2g", split="train")
obs_dict, _ = env2.reset(seed=0)

is_home_history = {agent: [] for agent in env2.agents}
soc_history     = {agent: [] for agent in env2.agents}

while env2.agents:
    # Record current state
    for agent in list(env2.agents):
        if agent in obs_dict:
            obs = obs_dict[agent]
            soc_history[agent].append(float(obs[0]))
            is_home_history[agent].append(float(obs[1]) if len(obs) > 1 else 1.0)

    # Random actions
    actions = {a: env2.action_space[a].sample() for a in env2.agents}
    obs_dict, _, _, _, _ = env2.step(actions)

print(f"Episode length: {len(list(soc_history.values())[0])} steps")
print(f"EV agents tracked: {list(is_home_history.keys())}")

In [ ]:
# Plot: is_home and SOC for the first 3 EVs
agents_to_plot = list(soc_history.keys())[:3]
fig, axes = plt.subplots(len(agents_to_plot), 1, figsize=(13, 7), sharex=True)

for ax, agent in zip(axes, agents_to_plot):
    soc = np.array(soc_history[agent])
    home = np.array(is_home_history[agent])
    steps = np.arange(len(soc))

    # Shade away periods
    ax.fill_between(steps, 0, 1, where=home < 0.5, alpha=0.25, color='gray',
                    label='Away (masked)')
    # SOC trace
    ax.plot(soc, color='#1565C0', linewidth=1.5, label='SOC')
    ax.axhline(0.8, color='tomato', linewidth=1.2, linestyle='--',
               label='Departure min (0.8)')
    ax.axhline(0.1, color='orange', linewidth=0.8, linestyle=':', label='SOC_min (0.1)')

    ax.set_ylabel('SOC')
    ax.set_ylim(-0.05, 1.1)
    ax.set_title(f'{agent} — SOC and availability (random policy)')
    ax.legend(loc='upper right', fontsize=7, ncol=3)

axes[-1].set_xlabel('Timestep (1-hour intervals)')
plt.suptitle('Task 3 — EV availability mask + SOC trace (1 week, random policy)', y=1.01)
plt.tight_layout()
plt.show()

## 5. Departure Constraint Satisfaction Rate

Besides episode reward, Task 3 tracks `departure_success_rate` —
the fraction of departures where SOC ≥ 80% was achieved.
A well-trained policy must balance arbitrage profit *and* departure readiness.

In [ ]:
# Show the departure_success_rate from the eval_protocol
print("Task 3 evaluation protocol:")
for k, v in MARLEVTask.eval_protocol.items() if hasattr(MARLEVTask, 'eval_protocol') else []:
    print(f"  {k}: {v}")

# Illustrate the constraint: SOC must be >= 0.8 at each departure
print("\nKey constraint: SOC ≥ 0.8 required at each departure time.")
print("Failure cost (departure_penalty_weight = 5.0) is much higher than arbitrage gain.")
print("\n→ A good policy must prioritize: [safety first] → [then profit]")
print("  This is analogous to a safe RL problem with hard constraints.")

In [ ]:
# Visualize the tradeoff: arbitrage vs departure success
fig, ax = plt.subplots(figsize=(8, 5))

# Conceptual Pareto frontier
np.random.seed(42)
n_agents = 30

# Random policy: poor on both
random_success = np.random.uniform(0.3, 0.6, 10)
random_reward  = np.random.uniform(-500, -200, 10)

# Rule-based: decent success, moderate reward
rule_success = np.random.uniform(0.7, 0.9, 10)
rule_reward  = np.random.uniform(-100, 100, 10)

# Oracle: high success, high reward
oracle_success = np.random.uniform(0.92, 0.99, 5)
oracle_reward  = np.random.uniform(200, 400, 5)

ax.scatter(random_success, random_reward, s=60, color='tomato', alpha=0.7,
           label='Random policy', zorder=3)
ax.scatter(rule_success, rule_reward, s=60, color='steelblue', alpha=0.7,
           label='Rule-based', zorder=3)
ax.scatter(oracle_success, oracle_reward, s=80, color='#4CAF50', alpha=0.9,
           marker='*', label='Oracle (upper bound)', zorder=4)

ax.axvline(0.8, color='gray', linewidth=1.2, linestyle='--', alpha=0.7,
           label='Target: 80% departure success')

ax.set_xlabel('Departure success rate')
ax.set_ylabel('Episode reward')
ax.set_title('Task 3 — Tradeoff: departure safety vs arbitrage profit (conceptual)')
ax.legend()
plt.tight_layout()
plt.show()

print("Note: the actual numbers depend on your trained policy.")
print("A good RL agent should be in the top-right corner.")

---

## Summary — What Makes Task 3 Unique

| Feature | Details |
|---|---|
| **Availability mask** | Agent action = 0 when EV is away; `is_home` in observation |
| **Departure constraint** | SOC ≥ 80% required at each departure; heavy penalty if violated |
| **Extra metric** | `departure_success_rate` in evaluation protocol |
| **Episode horizon** | 168 steps = 1 week (longest of the three tasks) |
| **RL challenge** | Safe RL with intermittent availability + long-horizon planning |

## Next

→ [NB07 — Evaluation Protocol & Baseline Comparison](./nb07_evaluation.ipynb)